# RAG — Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)

_Papers · Transformers & LLMs_

**Let a generator read documents fetched on the fly, and average its answer over them.**

---

This notebook builds RAG one step at a time: the marginalization math by hand, a toy world whose answers live only in documents, a tiny attention reader, and a head-to-head against a model that never retrieves. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q river
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Marginalize the answer over retrieved documents by hand

RAG's core move is to *marginalize*: the answer probability is $p(y|x) = \sum_z p_\eta(z|x)\,p_\theta(y|x,z)$ — a weighted average of the generator's answer over each retrieved document $z$, weighted by how strongly the retriever ranked that document. We compute one such average by hand first: softmax the retriever scores into weights, then combine them with the generator's per-document answer probabilities.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# Worked example: marginalize the generator over top-2 docs.
scores3 = torch.tensor([2.0, 1.0, 0.5])
print("softmax over 3 scores:", [round(v,4) for v in torch.softmax(scores3,0).tolist()])

w = torch.softmax(torch.tensor([2.0, 1.0]), 0)        # top-2 retriever weights p(z|x)
g = torch.tensor([0.8, 0.1])                          # generator p(answer | x, z_j)
marg = (w * g).sum()                                  # p(y|x) = sum_z p(z|x) p(y|x,z)

print("top-2 retriever weights:", [round(v,4) for v in w.tolist()])
print("marginal p(y|x) = %.4f*%.1f + %.4f*%.1f = %.4f" % (w[0],g[0],w[1],g[1],marg))
# softmax over 3 scores: [0.6285, 0.2312, 0.1402]
# top-2 retriever weights: [0.7311, 0.2689]
# marginal p(y|x) = 0.7311*0.8 + 0.2689*0.1 = 0.6117

### Step 2 — Build the toy world and the encoders

We make a tiny knowledge task: countries, a pool of candidate capitals, and a document store with one passage per country (plus distractor passages). The key trick comes later — the country$\to$capital mapping is re-randomized each episode, so the answer lives only in the passages, never in the weights. Here we just build the vocabulary, a shared embedding table, and a bag-of-embeddings encoder used by both the retriever and the reader.

In [ ]:
# --- Toy world: countries, capitals, and a document store (one passage per fact). ---
countries = ["france","japan","egypt","peru","norway","kenya","brazil","canada"]
caps      = ["paris","tokyo","cairo","lima","oslo","nairobi","brasilia","ottawa",
             "marseille","kyoto","rio","toronto"]            # a pool to draw answers from
distract  = ["river nile flows","mount fuji tall","ocean wide blue","desert sand hot"]

toks = set(["capital"]) | set(countries) | set(caps)
for d in distract:
    toks |= set(d.split())

vocab = {t:i for i,t in enumerate(sorted(toks))}
V = len(vocab)

def enc(txt):
    return [vocab[t] for t in txt.split()]

cap_id = {c:i for i,c in enumerate(caps)}
A = len(caps)

D = 24
emb = nn.Embedding(V, D)                                # shared retriever/reader embeddings

def bag(ids):
    return emb(torch.tensor(ids)).mean(0)               # bag-of-embeddings encoder

print("vocab size:", V, " candidate capitals:", A)

### Step 3 — The reader, the parametric-only baseline, and the retriever

Three pieces. The **reader** (`Reader`) is the generator: a tiny attention head where the query attends over the passage tokens and reads out an answer — it *can* use a document. The **parametric-only** baseline (`ParamOnly`) has the same answer head but sees only the question, no document — our control. The **retriever** scores documents by inner product with the query (maximum inner-product search) and softmaxes the top-$K$ scores into $p_\eta(z|x)$.

In [ ]:
# --- Generator: a tiny attention "reader". Query attends over the passage tokens. ---
class Reader(nn.Module):
    def __init__(self):
        super().__init__()
        self.q = nn.Linear(D, D)
        self.k = nn.Linear(D, D)
        self.v = nn.Linear(D, D)
        self.out = nn.Linear(D, A)
    def forward(self, qv, doc_ids):
        toks = emb(torch.tensor(doc_ids))               # (L, D) passage token embeddings
        att  = torch.softmax(self.k(toks) @ self.q(qv) / D**0.5, 0)   # (L,) attention
        ctx  = (att.unsqueeze(1) * self.v(toks)).sum(0) # (D,) read-out
        return self.out(ctx)                            # (A,) answer logits

gen = Reader()

# --- Parametric-only baseline: same answer head, but reads ONLY the question. ---
emb2 = nn.Embedding(V, D)

def bag2(ids):
    return emb2(torch.tensor(ids)).mean(0)

class ParamOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D,32), nn.ReLU(), nn.Linear(32,A))
    def forward(self, qv):
        return self.net(qv)

po = ParamOnly()

# --- Retriever: inner-product scores over the document store, softmax the top-K. ---
K = 3

def build_docs(mapping):
    return [f"{c} capital {mapping[c]}" for c in countries] + distract

def retrieve(country, doc_vecs, k):
    q = bag(enc(country))
    sc = doc_vecs @ q                                   # inner products = MIPS scores
    tv, ti = torch.topk(sc, k)                          # top-K documents
    return q, ti, torch.softmax(tv, 0)                  # p_eta(z|x) over the top-K

def random_map():
    pool = caps[:]
    np.random.shuffle(pool)
    return {c: pool[i] for i,c in enumerate(countries)}

print("reader, baseline, and retriever ready (top-K =", K, ")")

### Step 4 — Train RAG and the baseline side by side

Each episode draws a fresh random country$\to$capital mapping, so the only way to be right is to read the current store. RAG's loss is the negative log of the **marginal** $p(y|x)$ — retrieve the top-$K$, then sum the generator's answer over them weighted by the retriever. The parametric-only baseline trains on the same targets but sees only the question. Training both together makes the later comparison apples-to-apples.

In [ ]:
# --- TRAIN. Each episode re-randomizes the mapping, so the answer must be RETRIEVED. ---
optR = torch.optim.Adam(list(gen.parameters()) + list(emb.parameters()), lr=5e-3)
optP = torch.optim.Adam(list(po.parameters()) + list(emb2.parameters()), lr=5e-3)

for ep in range(1200):
    m = random_map()
    docs = build_docs(m)

    # RAG: retrieve top-K, then MARGINALIZE the generator over them (the novel step).
    optR.zero_grad()
    dv = torch.stack([bag(enc(d)) for d in docs])
    loss = 0.0
    for c in countries:
        q, ti, w = retrieve(c, dv, K)
        py = 0.0
        for j in range(K):                             # p(y|x) = sum_z p(z|x) p(y|x,z)
            py = py + w[j] * torch.softmax(gen(q, enc(docs[ti[j]])), 0)
        loss = loss - torch.log(py[cap_id[m[c]]] + 1e-9)
    loss.backward()
    optR.step()

    # Parametric-only: same target, but it only sees the question (no document).
    optP.zero_grad()
    loss2 = 0.0
    for c in countries:
        loss2 = loss2 + F.cross_entropy(po(bag2(enc(c))).unsqueeze(0), torch.tensor([cap_id[m[c]]]))
    loss2.backward()
    optP.step()

print("training done (1200 episodes)")

### Step 5 — Evaluate on fresh document stores

We test on 50 brand-new random stores (400 queries total), each with a mapping never seen in training. RAG retrieves and marginalizes to predict; the baseline answers from the question alone. Because the answer lives only in the retrieved passage, RAG should be near-perfect while the parametric-only model sits near the random-guess floor of $1/A$.

In [ ]:
# --- EVALUATE on 50 FRESH random document stores (= 400 queries). ---
np.random.seed(123)

def rag_pred(c, dv, docs):
    q, ti, w = retrieve(c, dv, K)
    py = 0.0
    for j in range(K):
        py = py + w[j] * torch.softmax(gen(q, enc(docs[ti[j]])), 0)
    return caps[int(py.argmax())]

def po_pred(c):
    return caps[int(po(bag2(enc(c))).argmax())]

rag_ok = po_ok = tot = 0
for t in range(50):
    m = random_map()
    docs = build_docs(m)
    dv = torch.stack([bag(enc(d)) for d in docs])
    for c in countries:
        rag_ok += (rag_pred(c, dv, docs) == m[c])
        po_ok += (po_pred(c) == m[c])
        tot += 1

print("\nEval on 50 fresh document stores (8 countries each = %d queries):" % tot)
print("  with retrieval (RAG)          : %d/%d = %.3f" % (rag_ok, tot, rag_ok/tot))
print("  without retrieval (param-only): %d/%d = %.3f" % (po_ok, tot, po_ok/tot))
print("  random-guess baseline         : 1/%d = %.3f" % (A, 1/A))
# with retrieval (RAG)          : 400/400 = 1.000   <- reads the answer from the passage
# without retrieval (param-only): 39/400  = 0.098   <- near the random-guess floor
# random-guess baseline         : 1/12    = 0.083
# (Our small run, not the paper's reported numbers. The answer lives in the retrieved passage,
#  so the parametric-only model cannot get it; retrieval makes it solvable.)

## Visualize the data & results

_On a knowledge task whose answer lives only in a retrieved passage, how does accuracy compare with retrieval (RAG) versus without it (parametric-only)?_

### Step 1 — Re-create the world, reader, baseline, and retriever (standalone)

This visualization cell stands on its own, rebuilding the vocabulary, embeddings, `Reader`, parametric-only baseline, and retriever exactly as above (just written more compactly). Re-running it gives a clean, reproducible setup for the head-to-head.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

countries = ["france","japan","egypt","peru","norway","kenya","brazil","canada"]
caps = ["paris","tokyo","cairo","lima","oslo","nairobi","brasilia","ottawa",
        "marseille","kyoto","rio","toronto"]
distract = ["river nile flows","mount fuji tall","ocean wide blue","desert sand hot"]

toks = set(["capital"]) | set(countries) | set(caps)
for d in distract:
    toks |= set(d.split())

vocab = {t:i for i,t in enumerate(sorted(toks))}
V = len(vocab)

def enc(t):
    return [vocab[w] for w in t.split()]

cap_id = {c:i for i,c in enumerate(caps)}
A = len(caps)
D = 24
K = 3

emb = nn.Embedding(V,D)
def bag(ids):
    return emb(torch.tensor(ids)).mean(0)

class Reader(nn.Module):
    def __init__(s):
        super().__init__()
        s.q=nn.Linear(D,D); s.k=nn.Linear(D,D); s.v=nn.Linear(D,D); s.out=nn.Linear(D,A)
    def forward(s,qv,d):
        t=emb(torch.tensor(d))
        a=torch.softmax(s.k(t)@s.q(qv)/D**0.5,0)
        return s.out((a.unsqueeze(1)*s.v(t)).sum(0))
gen=Reader()

emb2=nn.Embedding(V,D)
def bag2(ids):
    return emb2(torch.tensor(ids)).mean(0)

class PO(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(D,32),nn.ReLU(),nn.Linear(32,A))
    def forward(s,qv):
        return s.net(qv)
po=PO()

def docs_of(m):
    return [f"{c} capital {m[c]}" for c in countries]+distract

def retrieve(c,dv,k):
    q=bag(enc(c)); sc=dv@q; tv,ti=torch.topk(sc,k)
    return q,ti,torch.softmax(tv,0)

def rmap():
    p=caps[:]; np.random.shuffle(p)
    return {c:p[i] for i,c in enumerate(countries)}

print("setup rebuilt")

### Step 2 — Train RAG and the parametric-only baseline

Same training loop as the reference: 1200 episodes, fresh random mapping each time. RAG minimizes the negative log marginal over the retrieved top-$K$; the baseline minimizes cross-entropy on the question alone. Both update together so they share the same training budget.

In [ ]:
optR=torch.optim.Adam(list(gen.parameters())+list(emb.parameters()),lr=5e-3)
optP=torch.optim.Adam(list(po.parameters())+list(emb2.parameters()),lr=5e-3)
for ep in range(1200):
    m=rmap(); ds=docs_of(m)
    optR.zero_grad(); dv=torch.stack([bag(enc(d)) for d in ds]); L=0.0
    for c in countries:
        q,ti,w=retrieve(c,dv,K); py=0.0
        for j in range(K):
            py=py+w[j]*torch.softmax(gen(q,enc(ds[ti[j]])),0)
        L=L-torch.log(py[cap_id[m[c]]]+1e-9)
    L.backward(); optR.step()
    optP.zero_grad(); L2=0.0
    for c in countries:
        L2=L2+F.cross_entropy(po(bag2(enc(c))).unsqueeze(0),torch.tensor([cap_id[m[c]]]))
    L2.backward(); optP.step()

print("training done")

### Step 3 — Evaluate both on fresh stores and print the gap

Evaluate on 50 fresh random stores and tally RAG vs parametric-only vs the random-guess floor. The printed numbers are the punchline: with retrieval the model is near-perfect; without it, it sits at chance — because the answer was placed only in the retrieved passage.

In [ ]:
np.random.seed(123)
rag_ok=po_ok=tot=0
for t in range(50):
    m=rmap(); ds=docs_of(m); dv=torch.stack([bag(enc(d)) for d in ds])
    for c in countries:
        q,ti,w=retrieve(c,dv,K); py=0.0
        for j in range(K):
            py=py+w[j]*torch.softmax(gen(q,enc(ds[ti[j]])),0)
        rag_ok+=(caps[int(py.argmax())]==m[c])
        po_ok+=(caps[int(po(bag2(enc(c))).argmax())]==m[c]); tot+=1
print("RAG (with retrieval)   :", rag_ok, "/", tot, "=", round(rag_ok/tot,3))
print("Parametric-only        :", po_ok, "/", tot, "=", round(po_ok/tot,3))
print("Random-guess baseline  : 1 /", A, "=", round(1/A,3))
# RAG (with retrieval)   : 400 / 400 = 1.0
# Parametric-only        : 39 / 400 = 0.098
# Random-guess baseline  : 1 / 12 = 0.083
# Our small run, not the paper's number. The answer lives in the retrieved passage.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The retrieval ablation. You have a working RAG. Now break retrieval: feed the
            generator a zero (empty) document vector for every query, everything else identical,
            and re-evaluate on fresh random document stores. What did you remove, and what do you
            expect to happen to accuracy?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace the retrieved document the generator reads with zeros (or a constant), so $p_\theta(y|x,z)$ no longer depends on any real passage. — _This severs the non-parametric memory: the generator can only use what is in its weights, which is nothing reliable because the mapping is re-randomized each store._
- Identify what is gone: the term $p_\theta(y \mid x, z)$ stops carrying document content, so the marginal $\sum_z p_\eta(z|x)\,p_\theta(y|x,z)$ degrades to a content-free guess. — _The retriever weights $p_\eta(z|x)$ still exist, but they multiply a generator that can no longer read the answer out of $z$._
- Re-evaluate: expect accuracy to collapse toward the random-guess baseline ($1/\text{number of capitals}$). — _With the mapping randomized per store, the answer exists only in the passage; remove the passage and there is nothing left to be right about._

**Answer:** You removed the non-parametric memory &mdash; the generator no longer reads
                 a real passage, so $p_\theta(y|x,z)$ is content-free and the marginal becomes a
                 guess. Accuracy collapses toward $1/A$ (random over $A$ capitals). This mirrors our
                 main result: with retrieval, the model is ~100% on the toy task; without it (the
                 parametric-only baseline) it sits near the random-guess floor, because the answer
                 lives in the retrieved passage, not in the weights.

</details>

**Problem 2.** Re-derive the worked example with a different generator. Keep the top-2 retriever
            weights $[0.7311, 0.2689]$, but now the generator assigns
            $p_\theta(y|x,z_0)=0.3$ and $p_\theta(y|x,z_1)=0.9$ &mdash; i.e. the answer is
            actually in the second (less-retrieved) document. Compute the marginal
            $p(y|x)$. What does the result reveal about retriever quality?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the marginalization: $p(y|x) = 0.7311\cdot 0.3 + 0.2689\cdot 0.9$. — _Same equation $p(y|x)=\sum_z p(z|x)p(y|x,z)$; only the per-document answer probabilities changed._
- Compute: $0.7311\cdot 0.3 = 0.2193$ and $0.2689\cdot 0.9 = 0.2420$, so $p(y|x) = 0.2193 + 0.2420 = 0.4613$. — _The answer-bearing document ($z_1$) is down-weighted to $0.27$, so its strong $0.9$ is muffled; the marginal lands at only $0.46$._
- Conclude: a good answer is wasted if the retriever ranks its document low. — _The generator can only contribute in proportion to $p_\eta(z|x)$; a mis-ranked retriever caps the achievable answer probability._

**Answer:** $p(y|x) = 0.7311\cdot 0.3 + 0.2689\cdot 0.9 = 0.2193 + 0.2420 = 0.4613$. Even
                 though one document supports the answer with probability $0.9$, the marginal is only
                 $0.46$ &mdash; far below the original $0.6117$ &mdash; because the retriever put most
                 weight ($0.73$) on the wrong document. The lesson: RAG is bottlenecked by the
                 retriever. If relevance ranking is poor, marginalizing cannot recover; this is why
                 the paper trains the query encoder end-to-end and uses a strong DPR retriever.

</details>

**Problem 3.** In our experiment the country&rarr;capital mapping is re-randomized every training
            episode, and the parametric-only baseline still scores near the random-guess floor.
            Why does randomizing the mapping make the comparison fair &mdash; and what would
            happen to the baseline if we had used a single fixed mapping the whole time?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reason about a fixed mapping: if "france" always maps to "paris," a parametric model can store that pair in its weights and answer without ever reading a passage. — _With a fixed mapping the answer is recoverable from the question alone, so retrieval is not actually necessary &mdash; the test would not isolate retrieval's contribution._
- Reason about a randomized mapping: when "france" maps to a different capital each store, no question&rarr;answer rule fits in fixed weights. — _The answer becomes a property of the current document store, not the question; only a model that reads the store can be right._
- Conclude: randomization forces the task to be genuinely knowledge-intensive, so the gap between RAG and parametric-only measures retrieval, not memorization. — _This mirrors the paper's setting: real-world facts the model never memorized, where looking them up is the only way._

**Answer:** Randomizing the mapping makes the answer depend on the current document
                 store rather than on the question, so memorization in weights cannot help &mdash; the
                 model must retrieve. That is exactly what isolates retrieval's contribution. With a
                 single fixed mapping, the parametric-only baseline would simply memorize all the
                 pairs and score ~100% too, and the experiment would prove nothing. By randomizing, we
                 force the task to be knowledge-intensive: in our run RAG reaches ~100% while the
                 parametric-only model stays near the random-guess floor of $1/A$.

</details>